# 03 — Predict no-shows and choose the model honestly (Part 3)
Input: `appointments_clean.csv` from Part 1. **The test set is only for reporting.** No model, setting or threshold is chosen with it.

Prediction point: the evening before the appointment, after that day's reminders were sent or skipped. So `reminder_sent` and `prior_no_shows` are both known when the score is made.

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, confusion_matrix, precision_score, recall_score,
                             f1_score, roc_auc_score)
import sklearn
print("scikit-learn", sklearn.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)

df = pd.read_csv("appointments_clean.csv")   # row order kept exactly as saved (sorted by appointment_id)
print(df.shape)

scikit-learn 1.9.1 | pandas 3.0.5 | numpy 2.5.3
(990, 16)


### Check of Task 2.2: pandas reproduces the SQL clinic counts
These are the same numbers as the SQL query in `02_queries.sql`, worked out independently with `groupby`.

In [2]:
clinics = pd.read_csv("clinics_enriched.csv")
check = (df.merge(clinics[["clinic_id", "clinic_name", "city"]], on="clinic_id")
           .groupby(["clinic_name", "city"])
           .agg(appointments=("no_show", "size"), no_shows=("no_show", "sum")))
check["no_show_pct"] = (100 * check["no_shows"] / check["appointments"]).round(1)
print(check.sort_values("no_show_pct", ascending=False))

sql_result = {  # copied from the SQL output (Task 2.2, without HAVING)
    "Gomti Nagar Care Point": (90, 30), "Hadapsar Care Centre": (143, 39), "Malviya Nagar Clinic": (142, 38),
    "Kothrud Family Clinic": (160, 36), "Vijay Nagar Clinic": (101, 21), "Edappally Medical Centre": (142, 23),
    "Sitabuldi Health Point": (82, 13), "Vaishali Nagar Clinic": (130, 18)}
match = all(tuple(check.loc[name].iloc[0][["appointments", "no_shows"]]) == v for name, v in sql_result.items())
print("\npandas counts match SQL counts:", match)

                                  appointments  no_shows  no_show_pct
clinic_name              city                                        
Gomti Nagar Care Point   Lucknow            90        30         33.3
Hadapsar Care Centre     Pune              143        39         27.3
Malviya Nagar Clinic     Jaipur            142        38         26.8
Kothrud Family Clinic    Pune              160        36         22.5
Vijay Nagar Clinic       Indore            101        21         20.8
Edappally Medical Centre Kochi             142        23         16.2
Sitabuldi Health Point   Nagpur             82        13         15.9
Vaishali Nagar Clinic    Jaipur            130        18         13.8

pandas counts match SQL counts: True


## Task 3.1 — Features, split and baseline

In [3]:
X = df[["age", "distance_km", "fee_inr", "lead_days", "prior_no_shows", "reminder_sent",
        "booking_channel", "visit_type"]].copy()
X["reminder_sent"] = X["reminder_sent"].map({"Yes": 1, "No": 0})
X = pd.get_dummies(X, columns=["booking_channel", "visit_type"])
y = df["no_show"]
print("X columns (", X.shape[1], "):", list(X.columns))

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Train:", X_train.shape[0], "rows | Test:", X_test.shape[0], "rows")
print("No-shows in test set:", int(y_test.sum()), f"| overall no-show rate: {round(100 * y.mean(), 1)}%")
baseline = (y_test == 0).mean()
print(f"Baseline accuracy (always 'Attended'): {baseline:.4f} ({int((y_test == 0).sum())} of {len(y_test)})")

X columns ( 11 ): ['age', 'distance_km', 'fee_inr', 'lead_days', 'prior_no_shows', 'reminder_sent', 'booking_channel_App', 'booking_channel_Phone', 'booking_channel_Walk-In', 'visit_type_Consultation', 'visit_type_Follow-up']
Train: 792 rows | Test: 198 rows
No-shows in test set: 49 | overall no-show rate: 22.0%
Baseline accuracy (always 'Attended'): 0.7525 (149 of 198)


## Task 3.2 — Logistic regression

In [4]:
scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)   # fit on training rows only
X_test_s = scaler.transform(X_test)

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_s, y_train)
pred_lr = lr.predict(X_test_s)
prob_lr = lr.predict_proba(X_test_s)[:, 1]

print("Accuracy :", round(accuracy_score(y_test, pred_lr), 3))
print("Confusion matrix [[TN, FP], [FN, TP]]:\n", confusion_matrix(y_test, pred_lr))
print("Precision:", round(precision_score(y_test, pred_lr), 3))
print("Recall   :", round(recall_score(y_test, pred_lr), 3))
print("F1       :", round(f1_score(y_test, pred_lr), 3))
print("ROC-AUC  :", round(roc_auc_score(y_test, prob_lr), 3))

Accuracy : 0.788
Confusion matrix [[TN, FP], [FN, TP]]:
 [[139  10]
 [ 32  17]]
Precision: 0.63
Recall   : 0.347
F1       : 0.447
ROC-AUC  : 0.789


In [5]:
coefs = pd.DataFrame({"feature": X.columns, "coef_scaled": lr.coef_[0], "train_sd": scaler.scale_})
print(coefs.round(3).to_string(index=False))

i = list(X.columns).index("reminder_sent")
c, sd = lr.coef_[0][i], scaler.scale_[i]
print(f"\nreminder_sent: coefficient {c:.3f}, SD of the feature {sd:.3f}")
print(f"np.exp(coef)      = {np.exp(c):.2f}  -> odds multiplied per ONE STANDARD DEVIATION of reminder_sent")
print(f"np.exp(coef / sd) = {np.exp(c / sd):.2f}  -> odds multiplied for the real No (0) -> Yes (1) change")
print(f"i.e. about {round(100 * (1 - np.exp(c / sd)))}% lower odds of a no-show with a reminder")

                feature  coef_scaled  train_sd
                    age       -0.254    16.656
            distance_km        0.253     6.052
                fee_inr        0.453   184.254
              lead_days        0.614     8.614
         prior_no_shows        0.650     0.969
          reminder_sent       -0.581     0.495
    booking_channel_App       -0.047     0.486
  booking_channel_Phone        0.216     0.481
booking_channel_Walk-In       -0.186     0.435
visit_type_Consultation        0.016     0.493
   visit_type_Follow-up       -0.016     0.493

reminder_sent: coefficient -0.581, SD of the feature 0.495
np.exp(coef)      = 0.56  -> odds multiplied per ONE STANDARD DEVIATION of reminder_sent
np.exp(coef / sd) = 0.31  -> odds multiplied for the real No (0) -> Yes (1) change
i.e. about 69% lower odds of a no-show with a reminder


**Why the two numbers differ:** the model was fitted on standardised features. So the raw coefficient describes a one-*standard-deviation* step in `reminder_sent` (SD ≈ 0.49), which is only about half of the real 0 → 1 step. Dividing by the SD rescales it to the full No → Yes change.

**Conclusion (an association, not a cause):** in this model, holding the other features constant, appointments where a reminder was sent have roughly 69% lower odds of a no-show (odds × ≈ 0.31). The data are observational, so this does not prove that reminders *cause* fewer no-shows. For example, clinics or patients who get reminders may differ in other ways. `lead_days` and `prior_no_shows` are the strongest positive (risk-raising) features.

## Task 3.3 — Decision tree and overfitting

In [6]:
for depth in [None, 3]:
    tree = DecisionTreeClassifier(random_state=42, max_depth=depth)
    tree.fit(X_train, y_train)
    tr, te = accuracy_score(y_train, tree.predict(X_train)), accuracy_score(y_test, tree.predict(X_test))
    print(f"max_depth={str(depth):4s} (actual depth {tree.get_depth():2d}): train acc {tr:.3f} | test acc {te:.3f} | gap {tr - te:.3f}")
    if depth == 3:
        tree3 = tree
print(f"Baseline accuracy for comparison: {baseline:.4f}")

max_depth=None (actual depth 15): train acc 1.000 | test acc 0.672 | gap 0.328
max_depth=3    (actual depth  3): train acc 0.822 | test acc 0.778 | gap 0.044
Baseline accuracy for comparison: 0.7525


**What the gap shows:** the unlimited tree scores 100% on training rows but falls *below* the do-nothing baseline on the test set. It has memorised noise in the training data. Limiting it to depth 3 shrinks the gap and generalises far better.

## Task 3.4 — Random forest and a comparison table

In [7]:
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)
pred_rf = rf.predict(X_test)
prob_rf = rf.predict_proba(X_test)[:, 1]
print("Train accuracy:", round(accuracy_score(y_train, rf.predict(X_train)), 3))
print("Test accuracy :", round(accuracy_score(y_test, pred_rf), 3))
print("Recall        :", round(recall_score(y_test, pred_rf), 3))
print("ROC-AUC       :", round(roc_auc_score(y_test, prob_rf), 3))
imp = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)
print("\nTop three feature importances:\n", imp.head(3).round(3))

Train accuracy: 1.0
Test accuracy : 0.747
Recall        : 0.306
ROC-AUC       : 0.706

Top three feature importances:
 lead_days      0.220
age            0.197
distance_km    0.195
dtype: float64


In [8]:
rows = []
for name, pred, prob in [("Logistic regression", pred_lr, prob_lr),
                         ("Decision tree (depth 3)", tree3.predict(X_test), tree3.predict_proba(X_test)[:, 1]),
                         ("Random forest (100 trees)", pred_rf, prob_rf)]:
    rows.append({"model": name, "accuracy": accuracy_score(y_test, pred),
                 "recall": recall_score(y_test, pred), "roc_auc": roc_auc_score(y_test, prob)})
comparison = pd.DataFrame(rows).round(3)
print(comparison.to_string(index=False))

                    model  accuracy  recall  roc_auc
      Logistic regression     0.788   0.347    0.789
  Decision tree (depth 3)     0.778   0.224    0.700
Random forest (100 trees)     0.747   0.306    0.706


**Which looks best (reporting only):** logistic regression has the highest test ROC-AUC, so it ranks risky bookings best. Its accuracy is also the best or level with the best. The random forest fits the training data perfectly but ranks test bookings worse, another sign of overfitting on a dataset this small. The depth-3 tree's accuracy is close, but its AUC is clearly lower. The formal choice is made in Task 3.5 using training data only.

## Task 3.5 — Tune, choose the model and the threshold, then test once

In [9]:
# (a) grid search on the training set (scoring = ROC-AUC)
grid = GridSearchCV(RandomForestClassifier(random_state=42),
                    {"n_estimators": [100, 200], "max_depth": [3, 5, 8]},
                    cv=5, scoring="roc_auc")
grid.fit(X_train, y_train)
print("best_params_:", grid.best_params_)
print("best_score_ (cross-validated ROC-AUC):", round(grid.best_score_, 3))

best_params_: {'max_depth': 3, 'n_estimators': 200}
best_score_ (cross-validated ROC-AUC): 0.767


In [10]:
# (b) choose the model with 5-fold cross-validated ROC-AUC on the TRAINING set only
lr_pipe = Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=1000))])
cv_scores = {
    "Logistic regression (pipeline)": cross_val_score(lr_pipe, X_train, y_train, cv=5, scoring="roc_auc").mean(),
    "Decision tree (depth 3)": cross_val_score(DecisionTreeClassifier(max_depth=3, random_state=42),
                                               X_train, y_train, cv=5, scoring="roc_auc").mean(),
    "Random forest (100 trees)": cross_val_score(RandomForestClassifier(n_estimators=100, random_state=42),
                                                 X_train, y_train, cv=5, scoring="roc_auc").mean(),
    "Tuned random forest (best_score_)": grid.best_score_,
}
cv_table = pd.Series(cv_scores).round(3).sort_values(ascending=False)
print(cv_table.to_string())
winner = cv_table.index[0]
print("\nWinner (highest CV ROC-AUC):", winner)
print("For information only — tuned forest test AUC:",
      round(roc_auc_score(y_test, grid.best_estimator_.predict_proba(X_test)[:, 1]), 3))

Logistic regression (pipeline)       0.774
Tuned random forest (best_score_)    0.767
Random forest (100 trees)            0.747
Decision tree (depth 3)              0.678

Winner (highest CV ROC-AUC): Logistic regression (pipeline)
For information only — tuned forest test AUC: 0.741


In [11]:
# (c) choose the threshold from out-of-fold TRAINING probabilities
oof = cross_val_predict(lr_pipe, X_train, y_train, cv=5, method="predict_proba")[:, 1]
th_rows = []
for t in [0.5, 0.35]:
    flag = (oof >= t).astype(int)
    th_rows.append({"threshold": t, "precision": precision_score(y_train, flag), "recall": recall_score(y_train, flag),
                    "f1": f1_score(y_train, flag), "flagged": int(flag.sum())})
th_table = pd.DataFrame(th_rows).round(3)
print(th_table.to_string(index=False))
THRESHOLD = float(th_table.loc[th_table["f1"].idxmax(), "threshold"])
print("\nChosen threshold (higher out-of-fold F1):", THRESHOLD)

 threshold  precision  recall    f1  flagged
      0.50      0.575   0.249 0.347       73
      0.35      0.513   0.467 0.489      154

Chosen threshold (higher out-of-fold F1): 0.35


**Why recall matters here:** a missed warning (false negative) costs an empty doctor slot. An extra call to a patient who would have come anyway (false positive) costs about a minute of front-desk time. So catching more real no-shows is worth some extra calls. At 0.35 the model catches nearly twice as many training-set no-shows as at 0.5, with only a small loss of precision.

In [12]:
# (d) fit the winner on the full training set and evaluate ONCE on the test set
lr_pipe.fit(X_train, y_train)
probs = lr_pipe.predict_proba(X_test)[:, 1]
flags = (probs >= THRESHOLD).astype(int)
print(f"Final model: logistic regression (pipeline), threshold {THRESHOLD}")
print("Test precision:", round(precision_score(y_test, flags), 3))
print("Test recall   :", round(recall_score(y_test, flags), 3))
print("Flagged       :", int(flags.sum()), "of", len(flags), "test bookings", f"(true no-shows caught: {int((flags & y_test.values).sum())} of {int(y_test.sum())})")
print("Test ROC-AUC  :", round(roc_auc_score(y_test, probs), 3))

Final model: logistic regression (pipeline), threshold 0.35
Test precision: 0.571
Test recall   : 0.571
Flagged       : 49 of 198 test bookings (true no-shows caught: 28 of 49)
Test ROC-AUC  : 0.789


In [13]:
# (e) save the test-set risk scores
out = pd.DataFrame({"appointment_id": df.loc[X_test.index, "appointment_id"].values,
                    "no_show": y_test.values,
                    "risk_prob": probs.round(4),
                    "flag": flags})
out.to_csv("test_risk_scores.csv", index=False)
print("Saved test_risk_scores.csv:", out.shape, "| rows with flag = 1:", int(out["flag"].sum()))
print(out.sort_values("risk_prob", ascending=False).head(10).to_string(index=False))

Saved test_risk_scores.csv: (198, 4) | rows with flag = 1: 49
appointment_id  no_show  risk_prob  flag
         A0635        1     0.9372     1
         A0544        1     0.9241     1
         A0293        1     0.9196     1
         A0950        1     0.8186     1
         A0996        0     0.7667     1
         A0067        0     0.7498     1
         A0769        1     0.7444     1
         A0364        0     0.7319     1
         A0806        1     0.7035     1
         A0560        0     0.7019     1


**How a clinic would really set the threshold:** start from how many follow-up calls the front desk can make in an evening. Lower the threshold until the daily flagged list fills that capacity, and weigh each extra call (about a minute) against the cost of an empty slot. 0.35 is a reasonable starting point, not a fixed rule.